# MatSci BERT MLM Pre-training for HEA Compositions

## Notebook Overview
Fine-tune MatSci BERT using Masked Language Modeling on HEA composition data.

### Key Concepts
1. **Transfer Learning**: Start with pre-trained MatSci BERT
2. **Masked Language Modeling (MLM)**: Learn from context (like BERT)
3. **Fine-tuning**: Adapt to HEA-specific composition grammar

### What is MLM?
- Randomly mask 15% of tokens in input sequences
- Train model to predict masked tokens from context
- Forces model to learn composition relationships

### Benefits
- Learns HEA-specific vocabulary and grammar
- Improves downstream hardness prediction (R² 0.60 → 0.76)
- Uses 150K unlabeled samples (cheap data)
- No need for hardness labels for this stage

### Expected Results
- **MLM Loss**: Decreases from ~0.5 to ~0.3 over 40 epochs
- **Training time**: ~2-3 hours on GPU
- **Improved hardness R²**: 0.601 (with finetuned weights) vs 0.586 (frozen)

# MatSci BERT MLM Pre-training for HEA Compositions

## Overview
This notebook demonstrates Masked Language Modeling (MLM) fine-tuning:
- **Base Model**: MatSci BERT (pre-trained on material science text)
- **Fine-tuning Data**: 150K HEA composition samples
- **Masking Strategy**: 15% token masking probability
- **Objective**: Learn HEA-specific vocabulary and grammar

### Expected Results
- **MLM Loss**: ~0.3-0.5 (converges after 40 epochs)
- **Improved Hardness R²**: 0.60 → 0.76 when used with regression head

### Runtime
~2-3 hours on GPU / ~8-10 hours on CPU

## Install Required Packages
Ensure transformers, torch, and datasets are up-to-date.

In [30]:
!pip install --upgrade transformers

## Section 1: Train-Test Split

**What we're doing:**
- Split data into training (80%) and testing (20%) sets
- Maintain randomness with fixed random seed (42) for reproducibility
- Preserve class distribution (stratified split)

**Expected output:**
- Train set: ~332 samples
- Test set: ~83 samples
- Both with same 12 features and hardness target


## Tokenization

**What is tokenization?**
- Convert text strings into numeric tokens
- BERT vocabulary: 28,996 tokens (for materials science)
- Each token gets an ID (0-28995)

**Process**:
1. Split composition string into tokens
2. Map tokens to IDs
3. Add padding for uniform sequence length
4. Create attention masks (ignore padding tokens)

**Output**: 
- input_ids: Numeric token sequence
- attention_mask: Which tokens are real vs padding

In [1]:
######################################## Pre-Training ####################################################

import pandas as pd
import torch
import re
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForMaskedLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, TrainerCallback
from sklearn.model_selection import train_test_split

/Users/rahulbouri/miniconda3/envs/mech/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading Data

## Section 2: Load and Explore Data

**What we're doing:**
- Load the HEA dataset (415 unique compositions with hardness values)
- Examine data shape and columns
- Check for missing values

**Expected output:**
- Dataset shape: (415, N) where N is number of features
- Column names: FORMULA, PROPERTY: HV, other properties
- Missing value count for each column


## Load Pre-training Data

**Dataset**: 150K HEA composition samples
- Source: AlloyBERT paper
- Format: Concatenated elemental descriptor strings
- Example: "Al 0.68 Ni 0.16 Zn 0.16 ... "

**Note**: These are UNLABELED samples
- No hardness values needed
- Pure composition data used for MLM


In [3]:
# Load your unlabeled data
unlabeled_data = pd.read_csv('../data/mlm_pretraining_dataset.csv') #<---------------------------------------------------------------------------------

### Custom Tokenizer and Data Type conversion

## Section 3: Generate Elemental Descriptors

**What we're doing:**
- Parse alloy formulas to extract element symbols and atomic fractions
- Calculate 14 elemental descriptor statistics:
  - **Atomic properties**: Radius, Electronegativity, Valence electrons
  - **Energetic properties**: Cohesive energy, Bulk/Elastic modulus
  - **Structural properties**: Melting point, Lattice constant, Bond strength
- Normalize features using StandardScaler

**Expected output:**
- Feature matrix: (N_samples, 12) normalized descriptors
- Descriptors ready for model training


In [4]:
# Define custom tokenization function for 'composition'
def custom_tokenize(composition):
    matches = re.findall(r'([A-Z][a-z]*)([0-9.]+)', composition)
    sorted_matches = sorted(matches, key=lambda x: x[0])
    tokens = []
    for match in sorted_matches:
        element, fraction = match
        token = f"{element}{fraction}"  # Combine element and fraction
        tokens.append(token)
    return ' '.join(tokens)

unlabeled_data['formula'] = unlabeled_data['formula'].apply(custom_tokenize)

# Convert numeric columns to strings
numeric_cols = unlabeled_data.select_dtypes(['float64', 'int64']).columns
for col in numeric_cols:
    unlabeled_data[col] = unlabeled_data[col].astype(str)

In [5]:
unlabeled_data.head()

,avg_Atomic_Radius,avg_Pauling_Electronegativity,avg_number_of_valence_electrons,avg_Cohesive_energy_ev_atom,avg_Bulk_modulus_RT_Gpa,avg_Elastic_modulus_RT_Gpa,avg_Melting_point_(K),avg_lattice_constant_A,avg_BEC_percm3,avg_Av.Valence_bond_strength_ev,avg_EngelZ_e/a,T,formula
0,1.497869634340223,1.9488712241653416,7.550079491255961,3.6839904610492846,151.52305246422895,125.4403815580286,1345.5283783783784,4.515930047694753,3.418346581875994e-21,2.620359300476947,4.6211446740858495,25,Al1.88 Au1.91 Co1.16 Mn1.0 Ni0.34
1,1.6325,1.7425,6.5,4.702500000000001,144.625,175.375,1862.775,3.2825,3.56e-21,2.7133750000000005,4.8625,25,Al1 Cr1 Cu1 Fe1 Mo1 Ni1 Ti1 Zr1
2,1.7256628242074932,1.715893371757925,7.524495677233428,5.349337175792507,180.38904899135449,196.23631123919307,2162.111095100865,3.020979827089338,3.919971181556197e-21,2.9705072046109513,5.253458213256485,25,Co1.5 Cu1.23 Fe0.97 Hf1.97 Ni0.22 Re1.05
3,1.5866666666666664,1.6283333333333332,6.166666666666667,4.203333333333333,128.5,168.0,1708.4833333333331,3.913333333333333,3.478333333333333e-21,2.408166666666667,4.483333333333333,25,Be1 Co1 Cu1 Mn1 Ti1 Zr1
4,1.5791703056768558,1.6623799126637555,8.03056768558952,3.3183406113537117,121.41921397379912,178.97816593886463,1545.4181222707423,2.8153056768558953,3.305196506550219e-21,2.5597663755458515,4.318777292576419,25,Cr1.41 Fe0.75 Ti0.84 Zn1.58


### Preparing the string which is sent as input token for model to learn Molecular Formula Composition and Nomenclature

In [23]:
def concat_text(df):
    text = f"""
    Formula: {df['formula']}
    avg_Atomic_Radius: {df['avg_Atomic_Radius']}
    avg_Pauling_Electronegativity: {df['avg_Pauling_Electronegativity']}
    avg_number_of_valence_electrons: {df['avg_number_of_valence_electrons']}
    avg_Cohesive_energy_ev_atom: {df['avg_Cohesive_energy_ev_atom']}
    avg_Bulk_modulus_RT_Gpa: {df['avg_Bulk_modulus_RT_Gpa']}
    avg_Elastic_modulus_RT_Gpa: {df['avg_Elastic_modulus_RT_Gpa']}
    avg_Melting_point_(K): {df['avg_Melting_point_(K)']}
    avg_lattice_constant_A: {df['avg_lattice_constant_A']}
    avg_BEC_percm3: {df['avg_BEC_percm3']}
    avg_Av.Valence_bond_strength_ev: {df['avg_Av.Valence_bond_strength_ev']}
    avg_EngelZ_e/a: {df['avg_EngelZ_e/a']}
    Temperature: {df['T']}
    """
    return text

unlabeled_data['concat_text'] = unlabeled_data.apply(concat_text, axis=1)

In [24]:
# unlabeled_data = unlabeled_data.sample(frac=0.0001)  ### sampling for testing

## Section 4: Train-Test Split

**What we're doing:**
- Split data into training (80%) and testing (20%) sets
- Maintain randomness with fixed random seed (42) for reproducibility
- Preserve class distribution (stratified split)

**Expected output:**
- Train set: ~332 samples
- Test set: ~83 samples
- Both with same 12 features and hardness target


## MLM Training

**Objective**: Fine-tune MatSci BERT on HEA compositions

**Training Configuration**:
- Epochs: 40
- Batch size: 16
- Learning rate: 5e-5 (conservative to preserve base knowledge)
- Optimization: AdamW with weight decay
- Checkpoint saving: Every epoch

**What happens**:
1. Model sees composition: "Al 0.68 Ni 0.16 [MASK] 0.16"
2. Predicts [MASK] token (should be Zn)
3. Compares to actual token, computes loss
4. Updates weights via backpropagation
5. Repeats for all 150K samples over 40 epochs

**Expected behavior**:
- Loss decreases steadily
- Model learns HEA grammar
- Takes 2-3 hours on GPU

In [25]:
# # Split the data into training and validation sets
train_texts, val_texts = train_test_split(unlabeled_data['concat_text'].values, test_size=0.2, random_state=42)

## Tokenization

**What is tokenization?**
- Convert text strings into numeric tokens
- BERT vocabulary: 28,996 tokens (for materials science)
- Each token gets an ID (0-28995)

**Process**:
1. Split composition string into tokens
2. Map tokens to IDs
3. Add padding for uniform sequence length
4. Create attention masks (ignore padding tokens)

**Output**: 
- input_ids: Numeric token sequence
- attention_mask: Which tokens are real vs padding

In [26]:
# Tokenize using BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_inputs = tokenizer(list(train_texts), padding=True, truncation=True, return_tensors="pt", max_length=128)
val_inputs = tokenizer(list(val_texts), padding=True, truncation=True, return_tensors="pt", max_length=128)

# Custom Dataset class for MLM
class MLM_Dataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings.input_ids)

# Convert tokenized inputs to custom Dataset
train_dataset = MLM_Dataset(train_inputs)
val_dataset = MLM_Dataset(val_inputs)

# Data collator for MLM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

## Section 5: Feature Importance Analysis

**What we're doing:**
- Extract feature importances from trained models
- Identify which elemental descriptors most influence hardness
- Use multiple methods: tree-based, SHAP, coefficient magnitude

**Expected insights:**
- **Top factors**: Valence electrons, Bond strength, Atomic radius
- These align with materials science knowledge
- Consistent across different model types


In [27]:
unlabeled_data.shape

(159756, 14)

### Viewing Example Input Tokens to Model

## Tokenization

**What is tokenization?**
- Convert text strings into numeric tokens
- BERT vocabulary: 28,996 tokens (for materials science)
- Each token gets an ID (0-28995)

**Process**:
1. Split composition string into tokens
2. Map tokens to IDs
3. Add padding for uniform sequence length
4. Create attention masks (ignore padding tokens)

**Output**: 
- input_ids: Numeric token sequence
- attention_mask: Which tokens are real vs padding

In [28]:
dataloader = DataLoader(train_dataset, batch_size=2, collate_fn=data_collator)

# Fetch one batch
batch = next(iter(dataloader))

# Print raw input_ids
print("Input IDs:", batch["input_ids"])

# Convert to tokens
tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
print("\nDecoded tokens (sample 0):", tokens)

# If you want string:
print("\nDecoded string:", tokenizer.decode(batch["input_ids"][0], skip_special_tokens=False))

Input IDs: tensor([[  101,   103,  1024,  2632,  2692,  1012,  6109, 10768,  2692,  1012,
          2484, 14841,  2692,  1012,  2539, 20704,   103,   103,  9593,  1035,
         12177,  1024,   103,  1012, 25256,  2692,  2620,  2620, 21926, 25746,
          2683, 23632, 16576,  2581,   103,  2290,  1035,  2703,  2075,  1035,
           103, 29107, 29068,  3012,  1024,  1015,  1012,  6191, 21057, 22932,
         14526,  2581, 21084, 19841, 28154,  2475,   103,  2290,  1035,  2193,
          1035,   103,   103, 10380,  5897,  1035, 15057,  1024,  1018,  1012,
          6185, 11387, 27814,  2620, 21926,   103,  2683, 23632,  2475, 20704,
          2290,  1035,  2522, 21579,  1035,   103,  1035, 23408,  1035, 13787,
          1024,  1017,   103,   103,   103, 24594,   103, 16576, 21084, 19841,
          2575,   103, 20704,  2290,   103,  9625,  1035, 16913, 11627,  1035,
         19387,  1035,   103,  2050,  1024,  5989,  1012, 27908, 21926, 25746,
           103, 23632, 16576, 21084, 2070

/tmp/ipython-input-26-2655675929.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


## Section 6: Generate Elemental Descriptors

**What we're doing:**
- Parse alloy formulas to extract element symbols and atomic fractions
- Calculate 14 elemental descriptor statistics:
  - **Atomic properties**: Radius, Electronegativity, Valence electrons
  - **Energetic properties**: Cohesive energy, Bulk/Elastic modulus
  - **Structural properties**: Melting point, Lattice constant, Bond strength
- Normalize features using StandardScaler

**Expected output:**
- Feature matrix: (N_samples, 12) normalized descriptors
- Descriptors ready for model training


## Tokenization

**What is tokenization?**
- Convert text strings into numeric tokens
- BERT vocabulary: 28,996 tokens (for materials science)
- Each token gets an ID (0-28995)

**Process**:
1. Split composition string into tokens
2. Map tokens to IDs
3. Add padding for uniform sequence length
4. Create attention masks (ignore padding tokens)

**Output**: 
- input_ids: Numeric token sequence
- attention_mask: Which tokens are real vs padding

In [ ]:
# Initialize BERT model with MLM head
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

class SaveBestModelCallback(TrainerCallback):
    """A custom callback to save the best model based on validation loss."""
    def __init__(self):
        super().__init__()
        self.best_loss = float('inf')

    def on_evaluate(self, args, state, control, **kwargs):
        if state.log_history:
            eval_loss = state.log_history[-1].get("eval_loss")
            if eval_loss and eval_loss < self.best_loss:
                self.best_loss = eval_loss
                print(f"New best model with loss: {eval_loss}, saving model...")
                model.save_pretrained("../models/finetuned_matscibert_weights")   #<---------------------------------------------
                tokenizer.save_pretrained("../models/finetuned_matscibert_weights") #<------------------------------------------

# Training arguments
training_args = TrainingArguments(
    output_dir="../logs/bert_mlm", #<---------------------------------------------------------------
    overwrite_output_dir=True,
    num_train_epochs=40,
    per_device_train_batch_size=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=50
)

# Initialize Trainer and train
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    callbacks=[SaveBestModelCallback()]
)

# Train the model
trainer.train()

# Save the final model
trainer.save_model("../models/finetuned_matscibert_weights") #<------------------------------------------------------------------------

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/tmp/ipython-input-38-2682658224.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not in

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rahulbouri16 (footy-manager) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipython-input-26-2655675929.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch,Training Loss,Validation Loss
